# ⚡ Módulo 16 - Notebook 03: Caso Integrador Geoespacial PySpark

## 🌍 Proyecto End-to-End de Análisis Espacial a Escala

**Libro:** Saliendo de lo Pandito  
**Módulo:** 16 - Proyectos Integradores y GitHub  
**Duración estimada:** 95 minutos  
**Dificultad:** 🔴 Avanzado  
**Plataforma:** Databricks Free Edition

---

## 🎯 Objetivos de aprendizaje

Al finalizar este notebook serás capaz de:

✅ **Integrar** GeoPandas + H3 + PySpark  
✅ **Procesar** millones de coordenadas  
✅ **Analizar** cobertura territorial  
✅ **Optimizar** rutas y zonas  
✅ **Crear** mapas interactivos de Big Data

---

## 📋 Pre-requisitos

* ✅ Módulos 09, 10, 11-14 completados
* ✅ Conocimiento de GeoPandas y H3
* ✅ Familiaridad con PySpark

---

## 📚 Contenido

1. Caso de Negocio: Expansión Territorial
2. Carga y Procesamiento Espacial
3. Análisis H3 Distribuido
4. Heatmaps a Gran Escala
5. Optimización de Cobertura
6. Dashboard Geoespacial

---

## 💡 Por qué importa

**Geoespacial + Big Data = decis iones territoriales:**

* 🏪 **Retail:** ¿Dónde abrir nueva tienda?
* 🚚 **Logística:** Optimizar zonas de reparto
* 📈 **Marketing:** Segmentación territorial
* 🌍 **Expansión:** Análisis de mercados

**El stack completo de datos espaciales**

In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import functions as F

print("💾 CARGANDO DATOS GEOESPACIALES")
print("="*70)

CATALOG = "pandito_ds"
SCHEMA = "default"

try:
    # Cargar datos con coordenadas H3
    df = spark.table(f"{CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3")
    
    print(f"\n✅ Datos geoespaciales cargados")
    print(f"   📊 Registros: {df.count():,}")
    print(f"   🗂️ Particiones: {df.rdd.getNumPartitions()}")
    print(f"   🏪 Sucursales: {df.select('sucursal_id').distinct().count()}")
    print(f"   🌐 Hexs H3: {df.select('hex_id').distinct().count()}")
    
    # Mostrar columnas espaciales
    print(f"\n🗺️ Columnas espaciales disponibles:")
    spatial_cols = ['lat', 'lon', 'hex_id', 'zona']
    for col in spatial_cols:
        if col in df.columns:
            print(f"   • {col}")
    
    # Muestra de datos
    print(f"\n📊 Muestra de datos geoespaciales:")
    df.select('sucursal_nombre', 'zona', 'lat', 'lon', 'hex_id', 'ventas').show(5, truncate=False)
    
    print(f"\n🎯 Este proyecto incluirá:")
    print(f"   1. Análisis H3 distribuido con PySpark")
    print(f"   2. Heatmap de ventas a escala")
    print(f"   3. Análisis de cobertura territorial")
    print(f"   4. Optimización de zonas de entrega")
    print(f"   5. Dashboard geoespacial interactivo")
    
    USAR_DATOS_REALES = True
    
except Exception as e:
    print(f"\n⚠️  No se pudo cargar la tabla de Unity Catalog")
    print(f"   Error: {e}")
    print(f"\n📝 Solución:")
    print(f"   1. Ejecuta primero: 00_05_Preparacion_Datos_Empresariales.ipynb")
    print(f"   2. Verifica que la tabla exista: {CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3")
    print(f"\n   Continuando con datos sintéticos...")
    
    df = None
    USAR_DATOS_REALES = False

print("\n" + "="*70)

## 📚 Geoespacial a Escala con PySpark

### 🌍 Stack Completo

**Combinación poderosa:**

```
GeoPandas (local, pequeño)
    +
H3 (indexación hexagonal)
    +
PySpark (distribuido, Big Data)
    =
Análisis Geoespacial a Escala
```

---

### 🗂️ Procesamiento Distribuido

**Pandas vs PySpark:**

| Operación | Pandas | PySpark |
|-----------|--------|--------|
| 1M coordenadas | Lento | Rápido |
| 10M coordenadas | Crash | Rápido |
| 100M coordenadas | Imposible | Rápido |

**Ejemplo:**
```python
# Pandas (local)
df_pandas['hex'] = df_pandas.apply(
    lambda row: h3.geo_to_h3(row['lat'], row['lon'], 9), 
    axis=1
)
# Lento en 1M+ filas

# PySpark (distribuido)
from pyspark.sql.functions import udf

@udf(StringType())
def geo_to_h3_udf(lat, lon):
    return h3.geo_to_h3(lat, lon, 9)

df_spark = df_spark.withColumn('hex', geo_to_h3_udf('lat', 'lon'))
# Rápido incluso en 100M+ filas
```

---

### 🎯 Caso de Uso: Heatmap de Ventas

**Objetivo:** Mapa de calor de ventas por hexágono H3.

**Pipeline:**

```python
# 1. Agrupar por hex
df_heatmap = df.groupBy('hex_id').agg(
    F.sum('ventas').alias('ventas_totales'),
    F.count('*').alias('transacciones')
)

# 2. Convertir a Pandas para visualizar
df_pandas = df_heatmap.toPandas()

# 3. Crear mapa con folium
import folium
from folium.plugins import HeatMap

# Obtener coordenadas de cada hex
df_pandas['lat'] = df_pandas['hex_id'].apply(lambda h: h3.h3_to_geo(h)[0])
df_pandas['lon'] = df_pandas['hex_id'].apply(lambda h: h3.h3_to_geo(h)[1])

# Crear mapa
m = folium.Map(location=[-32.89, -68.84], zoom_start=11)

# Agregar heatmap
heat_data = [[row['lat'], row['lon'], row['ventas_totales']] 
             for _, row in df_pandas.iterrows()]
HeatMap(heat_data).add_to(m)

m.save('heatmap.html')
```

---

### 📏 Análisis de Cobertura

**Pregunta:** ¿Qué zonas tienen poca cobertura?

**Approach:**

```python
# 1. Crear grid de hexs en toda la ciudad
import h3

# Bbox de Mendoza
min_lat, max_lat = -33.0, -32.8
min_lon, max_lon = -69.0, -68.7

# Generar todos los hexs posibles
todos_los_hexs = h3.polyfill(
    geo_json_polygon, 
    res=9, 
    geo_json_conformant=True
)

# 2. Identificar hexs con ventas
hexs_con_ventas = set(df.select('hex_id').distinct().toPandas()['hex_id'])

# 3. Hexs SIN ventas (oportunidad)
hexs_sin_cobertura = todos_los_hexs - hexs_con_ventas

print(f"Hexs sin cobertura: {len(hexs_sin_cobertura)}")
print(f"Oportunidad de expansión")
```

---

### 🚚 Optimización de Zonas de Reparto

**Objetivo:** Dividir ciudad en N zonas equilibradas.

**Approach: K-Means Clustering**

```python
from pyspark.ml.clustering import KMeans
from pyspark.ml.feature import VectorAssembler

# 1. Preparar features
assembler = VectorAssembler(inputCols=['lat', 'lon'], outputCol='features')
df_features = assembler.transform(df)

# 2. K-Means (N zonas)
kmeans = KMeans(k=5, seed=1)  # 5 zonas
model = kmeans.fit(df_features)

# 3. Asignar zona a cada hex
df_zonas = model.transform(df_features)

# 4. Verificar balance
df_zonas.groupBy('prediction').agg(
    F.count('*').alias('hexs'),
    F.sum('ventas').alias('ventas_totales')
).show()
```

---

### 🏆 Best Practices

**1️⃣ Usar H3 para agregación espacial**
```python
# Mejor que lat/lon directamente
df.groupBy('hex_id').agg(F.sum('ventas'))
```

**2️⃣ Particionar por zona geográfica**
```python
df.repartition('zona')  # Mejora joins espaciales
```

**3️⃣ Cache para operaciones repetidas**
```python
df_hexs = df.groupBy('hex_id').agg(...).cache()
```

**4️⃣ Broadcast para tablas pequeñas**
```python
from pyspark.sql.functions import broadcast
df.join(broadcast(df_small), 'hex_id')
```

---

### 📊 Dashboard Geoespacial

**Componentes:**

1. **Mapa base:** Folium/Plotly
2. **Heatmap:** Ventas por hex
3. **Marcadores:** Sucursales
4. **Polígonos:** Zonas de reparto
5. **Métricas:** KPIs por zona

**Stack:**
* PySpark: Procesamiento
* Pandas: Preparación final
* Plotly/Folium: Visualización
* Streamlit/Dash: Dashboard interactivo

In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import warnings
warnings.filterwarnings('ignore')

print("🌍 PROYECTO INTEGRADOR: GEOESPACIAL + PYSPARK")
print("="*70)

print(f"\nVersión de Pandas: {pd.__version__}")
print(f"Versión de NumPy: {np.__version__}")

try:
    print(f"Versión de Spark: {spark.version}")
except:
    print("⚠️  SparkSession no disponible")

print("\n🎯 En este proyecto aplicarás:")
print("  • H3 distribuido con PySpark UDFs")
print("  • Agregaciones espaciales a escala")
print("  • K-Means clustering geográfico")
print("  • Heatmaps de Big Data")
print("  • Análisis de cobertura territorial")

print("\n📖 Stack geoespacial:")
print("  - PySpark: Procesamiento distribuido")
print("  - H3: Indexación hexagonal")
print("  - GeoPandas: Geometrías (local)")
print("  - Folium/Plotly: Visualización")

print("\n🛠️ Métodos clave:")
print("  - df.groupBy('hex_id').agg()")
print("  - udf() para funciones H3")
print("  - KMeans para clustering")
print("  - broadcast() para joins espaciales")

print("\n" + "="*70)
print("✅ Listo para análisis geoespacial a escala")

In [0]:
# Código de inicialización de notebook reindexado
import pandas as pd
import numpy as np
print('Notebook reindexado listo para práctica en Databricks')